# Displays data and creates datasets for testing

Uses observed data: `observed.csv` and deterministic forecasts up to 2 days `deterministic.csv` as a basis

In [ ]:
import importlib
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import scipy
import plotly.graph_objects as go
import plotly.express as px
from performance import plotly_forecasting as go_d

In [ ]:
test_dataset_path = Path(r'tests\test_datasets_hourly')

obs = pd.read_csv(test_dataset_path / 'observed.csv', sep=';', index_col=0, parse_dates=True)
obs.columns = pd.Index(pd.to_timedelta(obs.columns), name='leadtime')
obs.index.name = 'event_datetime'
display(obs.head(5))

det_p = pd.read_csv(test_dataset_path / 'deterministic.csv', sep=';', index_col=0, parse_dates=True)
det_p.columns = pd.Index(pd.to_timedelta(det_p.columns), name='leadtime')
det_p.index.name = 'production_datetime'
display(det_p.iloc[:5,:5])

det = det_p.stack()
idx = det.index.to_frame(index=False)
idx.loc[:, 'event_datetime'] = idx['production_datetime'] + idx['leadtime']
det.index = pd.MultiIndex.from_frame(idx[['event_datetime', 'leadtime']])
det = det.unstack('leadtime')
display(det.iloc[:5,:5])

In [ ]:
det_path = test_dataset_path / 'det.parquet'
if det_path.exists():
    det_parquet = pd.read_parquet(det_path)
else:
    det_parquet = det.stack().to_frame(name='Q').dropna()
    idx = det_parquet.index.to_frame(index=False)
    idx['production_datetime'] = idx['event_datetime'] - idx['leadtime']
    det_parquet.index = pd.MultiIndex.from_frame(idx[['production_datetime', 'event_datetime', 'leadtime']])
    det_parquet = det_parquet.sort_index()

    det_parquet.to_parquet(det_path)

display(det_parquet.head(5))

In [ ]:
importlib.reload(go_d)

leadtimes = det_parquet.index.get_level_values('leadtime').unique()[::6]

fig = go.Figure()
go_d.plot_lt_deterministic(fig, det_parquet, leadtimes=leadtimes, colorscale=px.colors.sequential.Magma, showlegend=False)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")

In [ ]:
importlib.reload(go_d)

production_datetimes = det_parquet.index.get_level_values('production_datetime').unique()[::24]

fig = go.Figure()
go_d.plot_pd_deterministic(fig, det_parquet, production_datetimes=production_datetimes, colorscale=px.colors.sequential.Jet, color_loops=12, showlegend=False)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")

## Create probabilistic forecasts
Heteroscedastinc log-normal distribution whose standard deviation increases with leadtime.  
Created for several probability bands.

In [ ]:
prob_path = test_dataset_path / 'prob.parquet'
if prob_path.exists():
    prob_parquet = pd.read_parquet(prob_path)
else:
    probability_bands = pd.Index([0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99], name='non_exceedance')
    leadtimes = det_parquet.index.get_level_values('leadtime').unique()
    sigma_dict = {lt: s for lt, s in zip(leadtimes, np.linspace(0.02, 0.3, len(leadtimes)))}

    norm_tile = pd.DataFrame(np.tile(scipy.stats.norm(0, 1).ppf(probability_bands), (det_parquet.shape[0], 1)))
    sigmas = det_parquet.index.to_frame(index=False)[['leadtime']].map(lambda x: sigma_dict[x]).values
    ln_noise = np.exp(-0.5 * sigmas**2 + sigmas*norm_tile)

    prob = pd.DataFrame(np.tile(det_parquet.copy(), (1, len(probability_bands))),
                        index=det_parquet.index,
                        columns=pd.Index(probability_bands, name='non_exceedance'),
    )
    prob *= ln_noise.values
    prob_parquet = prob.stack().to_frame(name='Q')
    prob_parquet.to_parquet(prob_path)

display(prob_parquet)


In [ ]:
importlib.reload(go_d)

production_datetimes = prob_parquet.index.get_level_values("production_datetime").unique()[::24]

fig = go.Figure()
go_d.plot_pd_probabilistic(fig, prob_parquet, production_datetimes=production_datetimes, colors=px.colors.qualitative.G10, color_loops=10)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")
fig.show()

In [ ]:
importlib.reload(go_d)

leadtimes = prob_parquet.index.get_level_values('leadtime').unique()[::6]

fig = go.Figure()
go_d.plot_lt_probabilistic(fig, prob_parquet, leadtimes=[pd.Timedelta('6h'), pd.Timedelta('24h'), pd.Timedelta('48h')], colors=px.colors.sequential.haline)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")
fig.show()

## Create ensemble forecasts
Based on probabilistic forecasts sampled with autoregressive p-value sampling.  

Requires some functions for efficiency. These are defined below.

In [ ]:
def generate_p_values(df, p_nan=0.9, n_min=2, **kwargs):
    p_values = df.copy()
    p_values.loc[:, :] = np.random.uniform(0, 1, size=p_values.shape)

    mask = np.random.binomial(n=1, p=1-p_nan, size=p_values.shape)
    mask[:, [0, -1]] = 1

    while (mask.sum(axis=1)<n_min).any():
        idxs = np.where(mask.sum(axis=1)<n_min)[0]
        mask[idxs, 1:-2] = np.random.binomial(n=1, p=1-p_nan, size=mask[idxs, 1:-2].shape)

    p_values = p_values.mask(~mask.astype(bool))
    p_values = p_values.interpolate(axis=1, **kwargs)
    
    return p_values

def interp_rows_matrix(x_values, xp, fp_rows):
    '''
    Vectorized row-wise equivalent of:

        np.interp(x_values[row], xp, fp_rows[row],
                  left=fp_rows[row, 0],
                  right=fp_rows[row, -1])

    Parameters
    ----------
    x_values : array, shape (n_rows, n_targets)
        Values to interpolate at. In your case: p_values_n[ens].values

    xp : array, shape (n_bands,)
        Common x-grid. In your case: probability_bands

    fp_rows : array, shape (n_rows, n_bands)
        Row-specific y-values. In your case: ensemble_data.values

    Returns
    -------
    out : array, shape (n_rows, n_targets)
    '''
    
    x_values = np.asarray(x_values, dtype=float)
    xp = np.asarray(xp, dtype=float)
    fp_rows = np.asarray(fp_rows, dtype=float)

    n_rows, n_targets = x_values.shape
    row_idx = np.arange(n_rows)[:, None]

    # Find interval indices
    j = np.searchsorted(xp, x_values, side="right")

    # Handle left/right extrapolation like np.interp
    left_mask = j == 0
    right_mask = j == len(xp)

    # Clip to valid interpolation interval
    j = np.clip(j, 1, len(xp) - 1)

    j0 = j - 1
    j1 = j

    x0 = xp[j0]
    x1 = xp[j1]

    y0 = fp_rows[row_idx, j0]
    y1 = fp_rows[row_idx, j1]

    w = (x_values - x0) / (x1 - x0)

    out = y0 * (1 - w) + y1 * w

    # Match np.interp left/right behavior
    out[left_mask] = fp_rows[row_idx, 0][left_mask]
    out[right_mask] = fp_rows[row_idx, -1][right_mask]

    return out

In [ ]:
ens_path = test_dataset_path / 'ens.parquet'
if ens_path.exists():
    ens_parquet = pd.read_parquet(ens_path)
else:
    n_ensembles = 10
    p_nan=0.95
    # interpolate_kwargs = dict(
    #     method='quadratic',
    #     n_min=3,
    # )
    interpolate_kwargs = dict(
        method='linear',
    )

    probability_bands = prob_parquet.index.get_level_values('non_exceedance').unique()

    ensemble_data = prob_parquet.unstack('non_exceedance')
    # display(ensemble_data)

    p_values_shape = prob_parquet.unstack('non_exceedance').iloc[:, 0].droplevel('event_datetime')
    p_value_mask = p_values_shape.unstack('leadtime')*0+1
    p_values_n = [(p_value_mask * generate_p_values(p_value_mask, p_nan=p_nan, **interpolate_kwargs)).stack('leadtime').to_frame(name='p-value') for i in range(n_ensembles)]
    # display(p_values_n[0])

    ensembles = []
    for i0, pv in enumerate(p_values_n):
        tmp = pd.DataFrame(
            interp_rows_matrix(pv.to_numpy(dtype=float), np.asarray(probability_bands, dtype=float), ensemble_data.to_numpy(dtype=float)),
            index=pv.index,
            columns=pd.Index([i0], name='ensemble_member')
        )
        ensembles.append(tmp)

    ensembles = pd.concat(ensembles, axis=1)
    ensembles.index = ensemble_data.index
    ens_parquet = ensembles.stack('ensemble_member').to_frame(name='Q')

    ens_parquet.to_parquet(ens_path)

display(ens_parquet)

In [ ]:
importlib.reload(go_d)

production_datetimes = ens_parquet.index.get_level_values("production_datetime").unique()[::24]

fig = go.Figure()
go_d.plot_pd_ensemble(fig, ens_parquet, production_datetimes=production_datetimes, colors=px.colors.qualitative.G10, color_loops=10)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")
fig.show()